# Infinaxis System Setup
This notebook goes through how to set up the needed Axis for 3 multiaxis system:
- A 4 axis mandrel type printer with a bed rotating around X (type A)
- A 4 axis tilting nozzle printer with the nozzle rotating aroung Y (type B)
- A 5 axis tilting and rotating bed printer with the bed rotating aroung X and Z (type A and C, respectively)

As for terminology infinaxis follows the convention that rotational axes are Type A, B or C, based on if the rotate around X, Y or Z, respectively. In the case of a stacked chain of rotating axis the type depends on the orientation when the system is at its origin position. Hence an A-C chain means that when the A axis is at 0 then the second rotating axis rotates around Z, and is hence type C.

Please note that infinaxis always expects that your system has the options to move in X,Y and Z outside the additional axis you configure. If your system cannot move in all 3 axes, infinaxis can still be used, it is just up to you to design the toolpaths so that the missing axis or axes are static.

The movement in XYZ does not need to be realised through a linear carriage system. We have succesfully used infinaxis to combine a 6-axis robot with a rotating table. In this case the resulting XYZ movement from infinaxis was then further post processed by the inverse kinematics of the 6 axis robot. The role of infinaxis was to relate the rotating table and the robot.

In [ ]:
import lab.fullcontrol.infinaxis as fci
from math import sin, cos, tau

# Common print settings
EW = 0.4
EH = 0.2

print_settings = {'extrusion_width': EW,'extrusion_height': EH}

#### Mandrel (4-axis)
This printer has one additional A axis on the bed chain.

In [ ]:
# The head chain is empty as no additional axis are present on the tool-head side of the printer.
head_chain = []

# The extra A axis is included in the bed chain.
# For a mandrel type printer an offset could be added to conform to the radius of the mandrel. 
# But as this type of printer is often used for symmetric deposition around the mandrel as opposed to a singular point on the surface,
# it can be more beneficial to include the mandrel offset in the toolpath design as opposed to during axes setup.
bed_chain = [fci.Axis(name='A')] # If the axis name is one of the three types its type is automatically inferred. If a custom name is needed the 'type' argument can be used, see the infinaxis_controls tutorial for more info.

#Once the head and bed chain are configured they are used to configure the custom point.
#This point can now be used in a similar manner to the normal fc.Point, but it will have the additional arguments based on the axis names.
#This point is hence fully configured by using Point(x,y,z,a).
Point = fci.configure_point(head_chain, bed_chain)

gcode_controls = fci.GcodeControls(
    head_chain = head_chain,
    bed_chain = bed_chain,
    initialization_data=print_settings 
    )

# Simple cylinder demonstration path
r_mandrel = 50 # mandrel radius
z_start = EH # nozzle height above mandrel
r = r_mandrel + z_start # radius of the spiral based on mandrel radius and starting z.
density = 36 # number of points per revolution
layers = 10 # number of layers in the x direction
w = 3.6 # spacing of each line

steps = []
steps.append(fci.Printer(print_speed=2160))
for i in range(layers):
    for j in range(density):
        angle = 360*(i+j/density)
        steps.append(Point(x=((i+j/density)*w),y=r*sin(angle/360*tau), z=r*cos(angle/360*tau), a=angle))
# add last point to finish last turn
angle = 360 * layers
steps.append(Point(x=layers*w,y=r*sin(angle/360*tau), z=r*cos(angle/360*tau), a=angle))

for step in steps:
    if isinstance(step, Point):
        # color is a gradient from A=0 deg (blue) to A=360 (red)
        step.color = [((step.a%360)/360), 0, 1-((step.a%360)/360)]

fci.transform(steps, 'plot', fci.PlotControls(color_type='manual',style='tube', zoom=0.75), show_tips=False)

# As you can see from the G-code lines below only the X and A axes are needed to follow the toolpath.
# In this case the toolpath was designed with A being explicitly defined.
# It is also possible to define the toolpath just in XYZ initially
# and then post process it to find A based the wanted approach angle of the nozzle or similar.
# Such post processing is intentionally not part of infinaxis natively, but can be integrated by the user.
gcode = fci.transform(steps,'gcode',gcode_controls)
print('first ten gcode lines:\n' + '\n'.join(gcode.split('\n')[:10]))
print('')
print('final ten gcode lines:\n' + '\n'.join(gcode.split('\n')[-10:]))

#### Tilting nozzle (4-axis)
This printer has one additional B axis on the head chain with an offset as the tip of the nozzle is offset from the rotation axis.

**Note on the B-axis sign convention**  
The centre path is parameterised in the **XZ plane**, so increasing `tilt` follows `X -> Z`.

Using the right-hand rule, the cross product is `X x Z = -Y`, so increasing the path angle corresponds to rotation about the negative Y-axis. Infinaxis defines positive B rotation about **+Y** by default, so the required axis angle is `b = -tilt`. Alternatively the default orientation of the B-axis can be changed by using the `orientation` argument.

An equivalent A-axis case in the **YZ plane** does not require this reversal because `Y x Z = X`, so increasing the path angle already corresponds to positive rotation about the X-axis.

By default, all axes in Infinaxis follow the right-hand rule. However, the orientation of any axis can be reversed by the user where required to match a particular machine configuration or coordinate convention.

In [ ]:
# The head chain contains the B-axis with a z offset of -50 mm
# as the tip of the nozzle is 50 mm below the center of rotation of the axis.
# The accuracy of this offset is paramoung as even small errors can easily cause printing failures.
# While an offset in a single coordinate is used here for simplicity,
# even if the system is designed with an intended single coordinate offset,
# it is very likely that based on mounting inaccuracy a real offset will be (x=0.1,y=0,z=-49.5).
head_chain = [fci.Axis(name='B',offset=fci.Point(x=0,y=0,z=-50))]

# The bed chain is empty as no additional axes are present.
bed_chain = []

#Once the head and bed chain are configured they are used to configure the custom point.
#This point can now be used in a similar manner to the normal fc.Point, but it will have the additional arguments based on the axis names.
#This point is hence fully configured by using Point(x,y,z,b).
Point = fci.configure_point(head_chain, bed_chain)

gcode_controls = fci.GcodeControls(
    head_chain = head_chain,
    bed_chain = bed_chain,
    initialization_data=print_settings 
    )

steps = []

path_tilt = 90
tilt_pr_layer = 1
layers = int(path_tilt/tilt_pr_layer)+1
path_rad = 100
circle_rad = 5
circle_density = 36

for i in range(layers):
    tilt = i*tilt_pr_layer
    tilt_r = tilt/360*tau
    center = Point(x=path_rad*cos(tilt_r),y=0,z=path_rad*sin(tilt_r))
    for j in range(circle_density+1):
        angle = j/circle_density*tau
        dx = circle_rad*cos(angle)*cos(tilt_r)
        dy = circle_rad*sin(angle)
        dz = circle_rad*cos(angle)*sin(tilt_r)
        # The b axis has a negative sign compared to the path tilt due to sign convention, see above note.
        steps.append(Point(x=center.x+dx,y=center.y+dy,z=center.z+dz,b=-tilt))

for step in steps:
    if isinstance(step, Point):
        # color is a gradient from B=0 deg (blue) to B=90 (red)
        step.color = [abs(step.b) / 90, 0, 1 - abs(step.b) / 90]

fci.transform(steps, 'plot', fci.PlotControls(color_type='manual', style='tube', zoom=0.75), show_tips=False)

gcode = fci.transform(steps, 'gcode', gcode_controls)
print('first ten gcode lines:\n' + '\n'.join(gcode.split('\n')[:10]))
print('')
print('final ten gcode lines:\n' + '\n'.join(gcode.split('\n')[-10:]))


#### 5 axes printer (AC bed chain)
This printer has a chain of two rotational axes on the bed: an outer A-axis and an inner C-axis.

Infinaxis expects any chain of axes to be defined from the outermost axis (closest to the system frame) to the innermost axis (closest to either the print bed or nozzle).

In this example, there is a Z-offset between the A- and C-axes. This offset is therefore tied to the A-axis, since its position is affected by A rotation. The C-axis also has a Z-offset because the print bed is mounted above it.

Since the C-axis rotates about Z, its Z-offset is unaffected by its rotation. In this particular case, the two Z-offsets could therefore be combined into a single offset tied to either axis. However, this is a special case, and combining offsets is not recommended without careful consideration.

In this example, the offsets are instead assigned to each axis separately, which is the recommended approach.

**Note on Head and bed chain offsets**  
Rotary axes on the head and bed chains have opposite effects on the relative orientation between the tool and part: rotating the bed moves the part relative to the tool, while rotating the head moves the tool relative to the part.

Axis offsets introduce an additional translational effect. For the same axis rotation and the same offset direction, the offset-induced XYZ compensation of a head axis is opposite to that of a bed axis. Reversing the offset direction reverses this compensation again. For example, the offset-induced motion of an A-axis on the head with a negative Z-offset can therefore match that of an A-axis on the bed with a positive Z-offset.

This equivalence applies only to the offset-induced XYZ compensation. The complete kinematics are still different because a bed-mounted axis rotates the part geometry, whereas a head-mounted axis only rotates the tool chain.

Offsets should always describe the actual physical geometry of the machine. If the positive direction of a physical axis is opposite to the Infinaxis default, reverse the axis `orientation` rather than changing the offset to compensate. By default, Infinaxis follows the right-hand rule.

**Note on Axis Order**  
The order of axes in a chain is very important. In general, changing the order changes the resulting kinematics because each inner axis is transformed by the axes outside it. Only in very special cases can the axes be reordered without changing the inverse kinematics. It is therefore always recommended to adhere strictly to the physical order of the axes in the intended machine configuration.

In [ ]:
# The head chain is empty as no additional axes are present.
head_chain = []

# The bed chain has the A-C chain with Z-offsets for both. C is placed last in the chain as it depends on A!
bed_chain = [fci.Axis(name='A',offset=fci.Point(x=0,y=0,z=15)),fci.Axis(name='C',offset=fci.Point(x=0,y=0,z=5))]

#Once the head and bed chain are configured they are used to configure the custom point.
#This point can now be used in a similar manner to the normal fc.Point, but it will have the additional arguments based on the axis names.
#This point is hence fully configured by using Point(x,y,z,a,c).
Point = fci.configure_point(head_chain, bed_chain)

gcode_controls = fci.GcodeControls(
    head_chain = head_chain,
    bed_chain = bed_chain,
    initialization_data=print_settings 
    )

#Five axis demo path with a increase in A tilt by 1 degree pr layer.
density = 36
layers = 90
layer_gap = EH
r = 10
z = EH / 2
steps = []

for layer in range(layers):
    tilt = layer
    # As the tilt is seen from a vertical perspective, the change in radius is related to sin. and the z height is related to cos.
    dr = layer_gap * sin(tilt / 360 * tau)
    dz = layer_gap * cos(tilt / 360 * tau)
    for j in range(density):
        t = j / density
        angle = 360 * (layer + t)
        r_now = r + dr * t
        z_now = z + dz * t
        steps.append(Point(x=r_now * sin(angle / 360 * tau), y=r_now * cos(angle / 360 * tau), z=z_now, a=tilt, c=angle))
    r += dr
    z += dz

for step in steps:
    if isinstance(step, Point):
        # color is a gradient from A=0 deg (blue) to A=90 (red)
        step.color = [abs(step.a) / 90, 0, 1 - abs(step.a) / 90]

fci.transform(steps, 'plot', fci.PlotControls(color_type='manual', style='tube', zoom=0.75), show_tips=False)

gcode = fci.transform(steps, 'gcode', gcode_controls)
print('first ten gcode lines:\n' + '\n'.join(gcode.split('\n')[:10]))
print('')
print('final ten gcode lines:\n' + '\n'.join(gcode.split('\n')[-10:]))
